# Titanic Ticket Purchasing System

## Project Overview

This project implements a Titanic ticket purchasing system based on the Titanic dataset.

The program:
- Loads and validates the Titanic dataset.
- Receives passenger details from the user.
- Validates `name`, `age`, `sex`, and `fare`.
- Determines passenger class from the fare.
- Generates a unique 6-digit ticket number.
- Estimates survival chance from historical Titanic data.
- Writes the final ticket to a text file.
- Prints a final message to the user.

## Main Assignment Rules

The solution follows these project requirements:
- Fare must be between the minimum and maximum valid fares in the dataset.
- Zero fares and missing fares are ignored when computing fare ranges.
- Age must be between 0 and 130.
- Sex must be either `male` or `female`.
- Name is stripped from leading and trailing spaces.
- If a fare matches multiple class ranges, the better class is selected.

## Imports and Constants

In this section, I import the required libraries and define constants used across the project.

Using constants improves readability and avoids hardcoded values in multiple places.

In [1]:
from __future__ import annotations

import random
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import pandas as pd

In [2]:
# Input dataset and output file paths
DATASET_PATH = Path("titanic.csv")
OUTPUT_PATH = Path("ticket.txt")

# Required dataset columns
REQUIRED_COLUMNS: tuple[str, ...] = (
    "pclass",
    "survived",
    "name",
    "sex",
    "age",
    "fare",
    "ticket",
)

# Allowed values for sex
ALLOWED_SEX: tuple[str, ...] = ("male", "female")

# Legal age range
MIN_AGE = 0.0
MAX_AGE = 130.0

# Ticket number rules
TICKET_DIGITS = 6
TICKET_MIN = 10 ** (TICKET_DIGITS - 1)
TICKET_MAX = 10 ** TICKET_DIGITS - 1
MAX_TICKET_ATTEMPTS = 10_000

# Age groups for survival calculation
AGE_SPLIT = 18.0
AGE_GROUP_MINOR = "under_18"
AGE_GROUP_ADULT = "18_and_over"

# Default ticket formatting width
TICKET_CELL_WIDTH = 25

## Data Classes

I use dataclasses to represent:
- validated passenger input
- the final ticket data

This makes the code more organized and easier to understand.

In [3]:
@dataclass(frozen=True)
class PassengerInput:
    """Validated passenger information collected from the user."""
    name: str
    age: float
    sex: str
    fare: float


@dataclass(frozen=True)
class Ticket:
    """Final ticket information after class and survival calculations."""
    passenger: PassengerInput
    pclass: int
    ticket_number: int
    survival_chance: float

## Custom Exception

This custom exception allows the program to exit cleanly if the user cancels input with `Ctrl+C` or `Ctrl+D`.

In [4]:
class UserCancelled(Exception):
    """Raised when the user stops the input process with Ctrl-C or EOF."""

## Dataset Loading and Cleaning

The dataset is loaded from titanic.csv and checked for required columns.

Important cleaning steps:
- numeric fields are converted safely
- `sex` is normalized to lowercase
- `ticket` values are converted to strings and stripped

In [5]:
def load_dataset(path: Path) -> pd.DataFrame:
    """
    Load the Titanic CSV file and validate that all required columns exist.
    """
    if not path.exists():
        raise FileNotFoundError(f"Dataset file not found: {path}")

    try:
        df = pd.read_csv(path, on_bad_lines="skip")
    except pd.errors.EmptyDataError as exc:
        raise ValueError(f"Dataset file is empty: {path}") from exc
    except pd.errors.ParserError as exc:
        raise ValueError(f"Dataset file is not a valid CSV file: {path}") from exc

    if df.empty:
        raise ValueError(f"Dataset has no rows: {path}")

    missing_columns = [column for column in REQUIRED_COLUMNS if column not in df.columns]
    if missing_columns:
        raise ValueError(f"Dataset is missing required columns: {missing_columns}")

    df = df.copy()

    df["fare"] = pd.to_numeric(df["fare"], errors="coerce")
    df["age"] = pd.to_numeric(df["age"], errors="coerce")
    df["pclass"] = pd.to_numeric(df["pclass"], errors="coerce")
    df["survived"] = pd.to_numeric(df["survived"], errors="coerce")

    df["sex"] = df["sex"].astype(str).str.strip().str.lower()
    df["ticket"] = df["ticket"].astype(str).str.strip()

    return df

In [6]:
def get_fare_bounds(df: pd.DataFrame) -> tuple[float, float]:
    """
    Return the minimum and maximum legal fare values.
    Zero fares and missing fares are ignored.
    """
    fares = df.loc[df["fare"] > 0, "fare"].dropna()

    if fares.empty:
        raise ValueError("Dataset contains no usable positive fare values.")

    return float(fares.min()), float(fares.max())


def get_class_fare_ranges(df: pd.DataFrame) -> dict[int, tuple[float, float]]:
    """
    Return the fare range for each passenger class.
    """
    usable = df[(df["fare"] > 0) & df["fare"].notna() & df["pclass"].notna()]

    ranges: dict[int, tuple[float, float]] = {}

    for pclass, group in usable.groupby("pclass"):
        fares = group["fare"]
        if not fares.empty:
            ranges[int(pclass)] = (float(fares.min()), float(fares.max()))

    if not ranges:
        raise ValueError("No class fare ranges could be computed from the dataset.")

    return ranges

## Load the Dataset

Now I load the real Titanic dataset and inspect its structure.

In [7]:
df = load_dataset(DATASET_PATH)

print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

Dataset shape: (1309, 14)
Columns: ['pclass', 'survived', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked', 'boat', 'body', 'home.dest']


,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


## Input Handling and Validation

This section contains helper functions for:
- safe input reading
- float parsing
- validating name, age, sex, and fare

Each input function keeps asking until a valid value is entered.

In [8]:
def _safe_input(prompt: str) -> str:
    """
    Read input safely and raise a custom exception on cancellation.
    """
    try:
        return input(prompt)
    except EOFError as exc:
        raise UserCancelled("end of input") from exc
    except KeyboardInterrupt as exc:
        raise UserCancelled("interrupted by user") from exc


def _parse_finite_float(raw: str) -> Optional[float]:
    """
    Convert a string to a finite float, or return None if invalid.
    """
    try:
        value = float(raw)
    except ValueError:
        return None

    if value != value or value in (float("inf"), float("-inf")):
        return None

    return value

In [9]:
def prompt_name() -> str:
    """
    Ask the user for a valid name.
    """
    while True:
        raw = _safe_input("Please enter your name: ")
        cleaned = raw.strip()

        if cleaned and any(ch.isalpha() for ch in cleaned):
            return cleaned

        print("Name is wrong. Please enter a valid name with at least one letter.")


def prompt_age() -> float:
    """
    Ask the user for a valid age.
    """
    while True:
        raw = _safe_input("Please enter your age: ").strip()
        value = _parse_finite_float(raw)

        if value is None:
            print("Age is wrong. Please enter a number between 0 and 130.")
            continue

        if not (MIN_AGE <= value <= MAX_AGE):
            print("Age is wrong. Please enter a number between 0 and 130.")
            continue

        return value


def prompt_sex() -> str:
    """
    Ask the user for a legal sex value.
    """
    while True:
        raw = _safe_input("Please enter your sex (male/female): ")
        cleaned = raw.strip().lower()

        if cleaned in ALLOWED_SEX:
            return cleaned

        print("Sex value is illegal. Please enter male or female.")


def prompt_fare(min_fare: float, max_fare: float) -> float:
    """
    Ask the user for a legal fare value inside the dataset range.
    """
    while True:
        raw = _safe_input(
            f"Please enter your fare (between {min_fare:.4f} and {max_fare:.4f}): "
        ).strip()

        value = _parse_finite_float(raw)

        if value is None:
            print(
                f"Fare payment is illegal. Please enter a number between "
                f"{min_fare:.4f} and {max_fare:.4f}."
            )
            continue

        if not (min_fare <= value <= max_fare):
            print(
                f"Fare payment is illegal. Please enter a value between "
                f"{min_fare:.4f} and {max_fare:.4f}."
            )
            continue

        return value

In [10]:
def collect_passenger_input(df: pd.DataFrame) -> PassengerInput:
    """
    Collect and validate all passenger fields.
    """
    min_fare, max_fare = get_fare_bounds(df)

    name = prompt_name()
    age = prompt_age()
    sex = prompt_sex()
    fare = prompt_fare(min_fare, max_fare)

    return PassengerInput(name=name, age=age, sex=sex, fare=fare)

## Class Inference

Passenger class is inferred from the fare paid.

If the fare fits more than one class range, the passenger is promoted to the better class:
- class 1 over class 2
- class 2 over class 3

In [11]:
def infer_pclass(fare: float, class_ranges: dict[int, tuple[float, float]]) -> int:
    """
    Infer passenger class from the fare.
    """
    matches = [
        pclass
        for pclass, (low, high) in class_ranges.items()
        if low <= fare <= high
    ]

    if matches:
        return min(matches)

    def distance(pclass: int) -> tuple[float, int]:
        low, high = class_ranges[pclass]

        if fare < low:
            return (low - fare, pclass)
        if fare > high:
            return (fare - high, pclass)
        return (0.0, pclass)

    return min(class_ranges, key=distance)

## Ticket Number Generation

The system creates a unique 6-digit ticket number.

Only existing 6-digit numeric ticket values from the dataset are considered for collision checking.

In [12]:
def get_existing_ticket_numbers(df: pd.DataFrame) -> set[int]:
    """
    Return all existing 6-digit numeric ticket numbers.
    """
    tickets = df["ticket"].astype(str).str.strip()
    six_digit_tickets = tickets[tickets.str.fullmatch(r"\d{6}")]

    return {int(ticket) for ticket in six_digit_tickets}


def generate_unique_ticket(existing: set[int]) -> int:
    """
    Generate a unique 6-digit ticket number.
    """
    if len(existing) >= (TICKET_MAX - TICKET_MIN + 1):
        raise RuntimeError("All 6-digit ticket numbers are already taken.")

    for _ in range(MAX_TICKET_ATTEMPTS):
        candidate = random.randint(TICKET_MIN, TICKET_MAX)
        if candidate not in existing:
            existing.add(candidate)
            return candidate

    raise RuntimeError("Failed to generate a unique ticket number.")

## Survival Chance Calculation

The survival probability is calculated from the dataset by grouping passengers according to:
- passenger class
- sex
- age group

The age groups are:
- `under_18`
- `18_and_over`

In [13]:
def age_group(age: float) -> str:
    """
    Convert age into one of the project age groups.
    """
    return AGE_GROUP_MINOR if age < AGE_SPLIT else AGE_GROUP_ADULT


def _add_age_group_column(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add an age_group column to a dataframe copy.
    """
    result = df.copy()
    result["age_group"] = pd.Series(pd.NA, index=result.index, dtype="object")

    has_age = result["age"].notna()
    result.loc[has_age & (result["age"] < AGE_SPLIT), "age_group"] = AGE_GROUP_MINOR
    result.loc[has_age & (result["age"] >= AGE_SPLIT), "age_group"] = AGE_GROUP_ADULT

    return result


def _mean_survived(subset: pd.DataFrame) -> Optional[float]:
    """
    Return the mean survival value for a subset.
    """
    if subset.empty:
        return None

    survived = subset["survived"].dropna()
    if survived.empty:
        return None

    return float(survived.mean())

In [14]:
def survival_chance(df: pd.DataFrame, pclass: int, sex: str, age: float) -> float:
    """
    Calculate the passenger's survival chance.
    """
    target_group = age_group(age)

    working = _add_age_group_column(
        df.dropna(subset=["survived", "sex", "pclass"])
    )

    grouped = working.groupby(
        ["pclass", "sex", "age_group"], dropna=False
    )["survived"].mean()

    key = (pclass, sex, target_group)

    if key in grouped.index:
        rate = grouped.loc[key]
        if pd.notna(rate):
            return float(rate)

    fallback_sets = [
        working[(working["pclass"] == pclass) & (working["sex"] == sex)],
        working[working["sex"] == sex],
        working,
    ]

    for subset in fallback_sets:
        rate = _mean_survived(subset)
        if rate is not None:
            return rate

    raise ValueError("Cannot compute survival chance from the dataset.")

## Ticket Formatting and File Output

The final ticket is written in a boxed layout and saved to `ticket.txt`.

In [15]:
def _format_cell(field: str, value: object, width: int = TICKET_CELL_WIDTH) -> str:
    """
    Format one ticket cell and pad it to a fixed width.
    """
    return f" {field}: {value}".ljust(width)


def _format_value(value: float) -> str:
    """
    Format numbers without unnecessary trailing zeros.
    """
    return f"{value:g}"


def format_ticket(ticket: Ticket) -> str:
    """
    Create the ticket text in the required boxed layout.
    """
    passenger = ticket.passenger

    rows = [
        ("ticket", f"{ticket.ticket_number:06d}", "fare", _format_value(passenger.fare)),
        ("age", _format_value(passenger.age), "class", ticket.pclass),
        ("sex", passenger.sex, "name", passenger.name),
    ]

    longest_cell = max(
        len(f" {field}: {value}")
        for left_field, left_value, right_field, right_value in rows
        for field, value in ((left_field, left_value), (right_field, right_value))
    )

    cell_width = max(TICKET_CELL_WIDTH, longest_cell + 1)
    total_width = 2 * cell_width + 3
    border = "-" * total_width

    lines = [border]

    for left_field, left_value, right_field, right_value in rows:
        left_cell = _format_cell(left_field, left_value, cell_width)
        right_cell = _format_cell(right_field, right_value, cell_width)
        lines.append(f"|{left_cell}|{right_cell}|")
        lines.append(border)

    return "\n".join(lines) + "\n"


def write_ticket_file(ticket: Ticket, path: Path) -> None:
    """
    Write the ticket to a text file.
    """
    try:
        path.write_text(format_ticket(ticket), encoding="utf-8")
    except OSError as exc:
        raise RuntimeError(f"Could not write ticket file {path}: {exc}") from exc

## Main Program Flow

The `main()` function connects all parts of the project:
1. Load dataset
2. Collect user input
3. Infer class
4. Generate ticket number
5. Estimate survival chance
6. Create and save the ticket
7. Print the final user message

In [16]:
def main() -> int:
    """
    Run the full Titanic ticket purchase flow.
    """
    try:
        df = load_dataset(DATASET_PATH)
        class_ranges = get_class_fare_ranges(df)

        print("Welcome to the Titanic ticket system.")
        print("Please enter your passenger details below.")

        passenger = collect_passenger_input(df)

        pclass = infer_pclass(passenger.fare, class_ranges)

        existing_tickets = get_existing_ticket_numbers(df)
        ticket_number = generate_unique_ticket(existing_tickets)

        chance = survival_chance(df, pclass, passenger.sex, passenger.age)

        ticket = Ticket(
            passenger=passenger,
            pclass=pclass,
            ticket_number=ticket_number,
            survival_chance=chance,
        )

        write_ticket_file(ticket, OUTPUT_PATH)

    except (FileNotFoundError, ValueError, RuntimeError) as exc:
        print(f"Error: {exc}", file=sys.stderr)
        return 1
    except UserCancelled as exc:
        print(f"\nCancelled ({exc}). No ticket was issued.", file=sys.stderr)
        return 130

    die_chance = (1 - chance) * 100

    print(f"Dear {passenger.name}, your chances to die on our trip are {die_chance:.1f}%.")
    print("Enjoy your trip and stay safe! :)")

    return 0

## Demonstration Without Interactive Input

In notebooks, fully interactive input is sometimes inconvenient to run during review.

For demonstration, I can simulate one passenger manually and show the final ticket.

In [17]:
class_ranges = get_class_fare_ranges(df)

sample_passenger = PassengerInput(
    name="Reem Mor",
    age=76,
    sex="male",
    fare=175.0,
)

sample_pclass = infer_pclass(sample_passenger.fare, class_ranges)
existing_tickets = get_existing_ticket_numbers(df).copy()
sample_ticket_number = generate_unique_ticket(existing_tickets)
sample_chance = survival_chance(df, sample_pclass, sample_passenger.sex, sample_passenger.age)

sample_ticket = Ticket(
    passenger=sample_passenger,
    pclass=sample_pclass,
    ticket_number=sample_ticket_number,
    survival_chance=sample_chance,
)

print(format_ticket(sample_ticket))
print(f"Estimated survival chance: {sample_chance:.2%}")
print(f"Estimated death chance: {(1 - sample_chance):.2%}")

-----------------------------------------------------
| ticket: 404206          | fare: 175               |
-----------------------------------------------------
| age: 76                 | class: 1                |
-----------------------------------------------------
| sex: male               | name: Reem Mor          |
-----------------------------------------------------

Estimated survival chance: 32.64%
Estimated death chance: 67.36%


In [18]:
write_ticket_file(sample_ticket, OUTPUT_PATH)
print(f"Ticket written to: {OUTPUT_PATH.resolve()}")

Ticket written to: C:\dev\amdocs-ai-course\homework\hw3\ticket.txt


## Conclusion

This notebook presents a clean and modular solution for the Titanic Ticket Purchasing System.

The solution follows the assignment requirements and uses the Titanic dataset for validation, class inference, and survival chance calculation.
